<a href="https://colab.research.google.com/github/aryangupta01/ML-Projects/blob/main/Text%20to%20Sql%20Finetune%20LLM%20Model/text_to_sql_finetuned_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text-to-SQL LORA Qwen3-0.6B Model Parameter Finetunning

## 1. Installation and Setup Libraries

In [ ]:
pip install -q trl evaluate peft sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from datasets import load_dataset
import torch
from torch.utils.data import Dataset
from tqdm.notebook import tqdm
import evaluate
from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM

from peft import get_peft_model, LoraConfig, TaskType

import pickle
import json
import matplotlib.pyplot as plt
import re

from urllib.request import urlopen
import io
from IPython.display import Markdown
def display_markdown(string):
    display(Markdown(string))

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 2. Data description

In [ ]:
dataset = load_dataset("b-mc2/sql-create-context", split="train")
dataset

README.md: 0.00B [00:00, ?B/s]

sql_create_context_v4.json:   0%|          | 0.00/21.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

Dataset({
    features: ['answer', 'question', 'context'],
    num_rows: 78577
})

In [ ]:
dataset[200]

{'answer': 'SELECT date_of_latest_revision FROM Catalogs GROUP BY date_of_latest_revision HAVING COUNT(*) > 1',
 'question': 'Find the dates on which more than one revisions were made.',
 'context': 'CREATE TABLE Catalogs (date_of_latest_revision VARCHAR)'}

In [ ]:
dataset = dataset.shuffle(seed=42).select(range(20000))
dataset

Dataset({
    features: ['answer', 'question', 'context'],
    num_rows: 20000
})

In [ ]:
dataset_split = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset_split['train']
test_dataset = dataset_split['test']
dataset_split

DatasetDict({
    train: Dataset({
        features: ['answer', 'question', 'context'],
        num_rows: 16000
    })
    test: Dataset({
        features: ['answer', 'question', 'context'],
        num_rows: 4000
    })
})

## 3. Model and Tokenizer

In [ ]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B").to(device)

In [ ]:
# Base model
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B").to(device)
print(model)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B", padding_side='left')
print(tokenizer)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layernorm): Qwe

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Qwen2TokenizerFast(name_or_path='Qwen/Qwen3-0.6B', vocab_size=151643, model_max_length=131072, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=

## 4. Preprocessing Data

In [ ]:
# Convert dataset to OAI messages
system_message = """You are an text to SQL query translator. Users will ask you questions in English and you will generate a SQL query based on the provided SCHEMA.
SCHEMA:
{schema}"""

def create_conversation_with_think(sample):
  return {
    "messages": [
      {"role": "system", "content": system_message.format(schema=sample["context"])},
      {"role": "user", "content": sample["question"]},
      {"role": "assistant", "content": f"<think>\n</think>\n\n{sample['answer']}"}
    ]
  }

def create_conversation_system_user(sample):
  return {
    "messages": [
      {"role": "system", "content": system_message.format(schema=sample["context"])},
      {"role": "user", "content": sample["question"]}
    ]
  }

In [ ]:
# Function to convert conversation to text format
def conversation_to_text(conversation):
    """Convert conversation format to text string for tokenization"""
    text = ""
    for message in conversation["messages"]:
        if message["role"] == "system":
            text += f"System: {message['content']}\n"
        elif message["role"] == "user":
            text += f"User: {message['content']}\n"
        elif message["role"] == "assistant":
            text += f"Assistant: {message['content']}\n"
    return text.strip()

# Alternative: Use chat template if your tokenizer supports it
def conversation_to_text_with_template(conversation, tokenizer):
    """Convert using tokenizer's chat template if available"""
    if hasattr(tokenizer, 'apply_chat_template'):
        return tokenizer.apply_chat_template(conversation["messages"], tokenize=False)
    else:
        return conversation_to_text(conversation)

In [ ]:
# Extract expected outputs (SQL queries only, without <think> tags)
# def extract_sql_from_response(response_text):
#     """Extract SQL query from response, removing <think> tags and extra text"""
#     # Remove <think>...</think> content
#     response_text = re.sub(r'<think>.*?</think>', '', response_text, flags=re.DOTALL)

#     # Clean up extra whitespace
#     response_text = response_text.strip()

#     # Remove any "assistant" prefix if present
#     if response_text.lower().startswith('assistant'):
#         response_text = response_text[9:].strip()

#     return response_text

def extract_sql_from_response(response_text):
    """Extract SQL query from response, removing <think> tags and extra text"""

    # Remove complete <think>...</think> blocks
    response_text = re.sub(r'<think>.*?</think>', '', response_text, flags=re.DOTALL)
    response_text = re.sub(r'.*?</think>', '', response_text, flags=re.DOTALL)

    # Remove standalone </think> tags
    response_text = re.sub(r'</think>', '', response_text)

    # Remove standalone <think> tags
    response_text = re.sub(r'<think>', '', response_text)

    # Clean up extra whitespace
    response_text = response_text.strip()

    # Remove any "assistant" prefix if present
    if response_text.lower().startswith('assistant'):
        response_text = response_text[9:].strip()

    # Remove any "assistant:" prefix if present
    if response_text.lower().startswith('assistant:'):
        response_text = response_text[10:].strip()

    return response_text

In [ ]:
expected_outputs = []
instructions_with_responses = test_dataset.map(create_conversation_with_think, remove_columns=test_dataset.features,batched=False)
instructions = test_dataset.map(create_conversation_system_user, remove_columns=test_dataset.features,batched=False)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

In [ ]:
for i in tqdm(range(len(instructions_with_responses))):
    # Convert conversations to text format
    full_conversation_text = conversation_to_text_with_template(instructions_with_responses[i], tokenizer)
    instruction_text = conversation_to_text_with_template(instructions[i], tokenizer)

    # Tokenize the text
    tokenized_instruction_with_response = tokenizer(
        full_conversation_text,
        return_tensors="pt",
        max_length=1024,
        truncation=True,
        padding=False
    )
    tokenized_instruction = tokenizer(instruction_text, return_tensors="pt")

    # Extract expected output
    expected_output = tokenizer.decode(
        tokenized_instruction_with_response['input_ids'][0][len(tokenized_instruction['input_ids'][0])-1:],
        skip_special_tokens=True
    )

    # Clean the expected output to get just the SQL
    clean_sql = extract_sql_from_response(expected_output)

    expected_outputs.append(clean_sql)

  0%|          | 0/4000 [00:00<?, ?it/s]

In [ ]:
print('## System' + ' User\n' + str(instructions[0]['messages']))
print('## System' + ' User' + ' Response\n' + str(instructions_with_responses[0]['messages']))
print('## Expected Response\n' + expected_outputs[0])

## System User
[{'content': 'You are an text to SQL query translator. Users will ask you questions in English and you will generate a SQL query based on the provided SCHEMA.\nSCHEMA:\nCREATE TABLE table_name_2 (record VARCHAR, attendance VARCHAR)', 'role': 'system'}, {'content': 'What was the score when attendance was 19,887?', 'role': 'user'}]
## System User Response
[{'content': 'You are an text to SQL query translator. Users will ask you questions in English and you will generate a SQL query based on the provided SCHEMA.\nSCHEMA:\nCREATE TABLE table_name_2 (record VARCHAR, attendance VARCHAR)', 'role': 'system'}, {'content': 'What was the score when attendance was 19,887?', 'role': 'user'}, {'content': '<think>\n</think>\n\nSELECT record FROM table_name_2 WHERE attendance = "19,887"', 'role': 'assistant'}]
## Expected Response
SELECT record FROM table_name_2 WHERE attendance = "19,887"


## 5. Testing the base model

In [ ]:
gen_pipeline = pipeline("text-generation",
                        model=model,
                        tokenizer=tokenizer,
                        device=device,
                        batch_size=2,
                        max_length=50,
                        truncation=True,
                        padding=False,
                        return_full_text=False)

Device set to use cuda


In [ ]:
tokenizer.padding_side = 'left'

text_inputs = [conversation_to_text_with_template(sample, tokenizer) for sample in instructions.select(range(20))]

with torch.no_grad():
    pipeline_iterator= gen_pipeline(text_inputs,
                                    max_length=512,
                                    num_beams=5,
                                    early_stopping=True)

generated_outputs_base = []
for text in tqdm(pipeline_iterator):
    generated_text = text[0]["generated_text"]
    clean_sql = extract_sql_from_response(generated_text)
    generated_outputs_base.append(clean_sql)

  0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
for i in range(3):
    print('@@@@@@@@@@@@@@@@@@@@')
    print('@@@@@ Instruction '+ str(i+1) +': ')
    print(instructions[i])
    print('\n\n')
    print('@@@@@ Expected response '+ str(i+1) +': ')
    print(expected_outputs[i])
    print('\n\n')
    print('@@@@@ Generated response '+ str(i+1) +': ')
    print(generated_outputs_base[i])
    print('\n\n')
    print('@@@@@@@@@@@@@@@@@@@@')

@@@@@@@@@@@@@@@@@@@@
@@@@@ Instruction 1: 
{'messages': [{'content': 'You are an text to SQL query translator. Users will ask you questions in English and you will generate a SQL query based on the provided SCHEMA.\nSCHEMA:\nCREATE TABLE table_name_2 (record VARCHAR, attendance VARCHAR)', 'role': 'system'}, {'content': 'What was the score when attendance was 19,887?', 'role': 'user'}]}



@@@@@ Expected response 1: 
SELECT record FROM table_name_2 WHERE attendance = "19,887"



@@@@@ Generated response 1: 
SELECT score FROM table_name_2 WHERE attendance = '19,887';



@@@@@@@@@@@@@@@@@@@@
@@@@@@@@@@@@@@@@@@@@
@@@@@ Instruction 2: 
{'messages': [{'content': 'You are an text to SQL query translator. Users will ask you questions in English and you will generate a SQL query based on the provided SCHEMA.\nSCHEMA:\nCREATE TABLE table_name_41 (pba_team VARCHAR, pick VARCHAR, player VARCHAR)', 'role': 'system'}, {'content': 'What is the PBA team for roberto jabar who was picked before number 1

## 6. Performing instruction fine-tuning with LoRA

In [ ]:
lora_config = LoraConfig(
    r=16,  # Low-rank dimension
    lora_alpha=32,  # Scaling factor
    target_modules=["q_proj", "v_proj"],  # Modules to apply LoRA
    lora_dropout=0.1,  # Dropout rate
    task_type=TaskType.CAUSAL_LM  # Task type should be causal language model
)

model = get_peft_model(model, lora_config)

In [ ]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 1024)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1024, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_fe

In [ ]:
# # Custom formatting function for your conversation style
# def formatting_conversations_func(dataset_batch):
#     """Format conversations for SFT training"""
#     conversations = []
#     for i in range(len(dataset_batch['question'])):
#         sample = {
#             'context': dataset_batch['context'][i],
#             'question': dataset_batch['question'][i],
#             'answer': dataset_batch['answer'][i]
#         }
#         conversation = create_conversation_with_think(sample)
#         conversation_text = conversation_to_text_with_template(conversation, tokenizer)
#         conversations.append(conversation_text)
#     return conversations

# Custom formatting function for your conversation style
def formatting_conversations_func(sample):
    """Format conversations for SFT training"""
    # SFTTrainer passes individual samples, not batches
    # Each sample has keys: 'answer', 'question', 'context'
    conversation = create_conversation_with_think(sample)
    conversation_text = conversation_to_text_with_template(conversation, tokenizer)
    return conversation_text

In [ ]:
# Print a few formatted examples
for i in range(3):
    print(f"Sample {i}:")
    print(formatting_conversations_func(train_dataset[i]))
    print("="*50)

Sample 0:
<|im_start|>system
You are an text to SQL query translator. Users will ask you questions in English and you will generate a SQL query based on the provided SCHEMA.
SCHEMA:
CREATE TABLE table_name_38 (score VARCHAR, home_team VARCHAR)<|im_end|>
<|im_start|>user
What is the score when parkgate is at home?<|im_end|>
<|im_start|>assistant
<think>

</think>

SELECT score FROM table_name_38 WHERE home_team = "parkgate"<|im_end|>

Sample 1:
<|im_start|>system
You are an text to SQL query translator. Users will ask you questions in English and you will generate a SQL query based on the provided SCHEMA.
SCHEMA:
CREATE TABLE table_12722302_2 (original_air_date VARCHAR, no VARCHAR)<|im_end|>
<|im_start|>user
What is the original air date for no. 2?<|im_end|>
<|im_start|>assistant
<think>

</think>

SELECT original_air_date FROM table_12722302_2 WHERE no = 2<|im_end|>

Sample 2:
<|im_start|>system
You are an text to SQL query translator. Users will ask you questions in English and you wi

In [ ]:
# # Fix the response template
# response_template = "<|im_start|>assistant"
# collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

# # Test with a sample
# sample_text = formatting_conversations_func(train_dataset[0])
# print("Sample text:")
# print(sample_text)
# print("\n" + "="*50 + "\n")

# # Tokenize correctly for the collator
# tokenized = tokenizer(
#     sample_text,
#     return_tensors="pt",
#     padding=False,
#     truncation=True,
#     max_length=1024
# )

# print("Original input_ids shape:", tokenized.input_ids.shape)
# print("Sample of input_ids:", tokenized.input_ids[0][:20])  # Show first 20 tokens

# # The collator expects a list of dictionaries, not a single dictionary
# # Convert the batch format to individual samples
# sample_for_collator = {
#     'input_ids': tokenized.input_ids[0],  # Remove batch dimension
#     'attention_mask': tokenized.attention_mask[0]  # Remove batch dimension
# }

# # Apply collator - it expects a list of samples
# try:
#     collated = collator([sample_for_collator])
#     print("\nAfter collator:")
#     print("Labels shape:", collated["labels"].shape)
#     print("Non-ignored labels (should be > 0):", (collated["labels"] != -100).sum().item())
#     print("Ignored labels:", (collated["labels"] == -100).sum().item())

#     # Show where the response starts
#     response_start = None
#     labels = collated["labels"][0]
#     for i, label in enumerate(labels):
#         if label != -100:
#             response_start = i
#             break

#     if response_start:
#         print(f"\nResponse starts at token index: {response_start}")
#         print("Response tokens (first 10):", labels[response_start:response_start+10])

#         # Decode to see what the response looks like
#         response_tokens = labels[labels != -100]
#         if len(response_tokens) > 0:
#             decoded_response = tokenizer.decode(response_tokens, skip_special_tokens=True)
#             print(f"\nDecoded response: {decoded_response}")
#     else:
#         print("\nWARNING: No response tokens found! This confirms the zero loss issue.")

# except Exception as e:
#     print(f"Error with collator: {e}")
#     print("This suggests the response_template is still not matching correctly.")

#     # Let's check what tokens the response_template produces
#     template_tokens = tokenizer.encode(response_template, add_special_tokens=False)
#     print(f"\nTemplate '{response_template}' tokens: {template_tokens}")

#     # Check if these tokens exist in our input
#     input_ids_list = tokenized.input_ids[0].tolist()
#     print(f"Template tokens found in input: {template_tokens[0] in input_ids_list}")

Sample text:
<|im_start|>system
You are an text to SQL query translator. Users will ask you questions in English and you will generate a SQL query based on the provided SCHEMA.
SCHEMA:
CREATE TABLE table_name_38 (score VARCHAR, home_team VARCHAR)<|im_end|>
<|im_start|>user
What is the score when parkgate is at home?<|im_end|>
<|im_start|>assistant
<think>

</think>

SELECT score FROM table_name_38 WHERE home_team = "parkgate"<|im_end|>



Original input_ids shape: torch.Size([1, 94])
Sample of input_ids: tensor([151644,   8948,    198,   2610,    525,    458,   1467,    311,   7870,
          3239,  45488,     13,  14627,    686,   2548,    498,   4755,    304,
          6364,    323])

After collator:
Labels shape: torch.Size([1, 94])
Non-ignored labels (should be > 0): 23
Ignored labels: 71

Response starts at token index: 71
Response tokens (first 10): tensor([   198, 151667,    271, 151668,    271,   4858,   5456,   4295,   1965,
          1269])

Decoded response: 
<think>

</thin

In [ ]:
# Data collator for completion-only training (only train on assistant responses)
response_template = "<|im_start|>assistant"
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

In [ ]:
# Training configuration
training_args = SFTConfig(
    output_dir="./sql_lora_output",
    num_train_epochs=2,
    save_strategy="epoch",
    fp16=True,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    max_seq_length=1024,
    do_eval=True,
    save_safetensors=False,
    logging_steps=500,
    eval_steps=1000,
    warmup_steps=100,
    learning_rate=2e-4,
    packing=False,
)

# Create trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    formatting_func=formatting_conversations_func,
    args=training_args,
    data_collator=collator
)

Applying formatting function to train dataset:   0%|          | 0/16000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/16000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/16000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/16000 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
# Train the model
print("Starting training...")
trainer.train()

# Save the model
torch.save(trainer.model.state_dict(), "./sql_lora_model.bin")
print("Model saved!")

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Starting training...


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aryan-gupta210302 (aryan-gupta210302-iit-kharagpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,0.215100
1000,0.070100
1500,0.055800
2000,0.051000
2500,0.053000
3000,0.045000
3500,0.035400
4000,0.039400
4500,0.037400
5000,0.043200


wandb: WARNING The get_url method is deprecated and will be removed in a future release. Please use `run.url` instead.


Model saved!


In [ ]:
# code to automatically download that saved model to the laptop

from google.colab import files
import os

# Specify the path to the saved model file
model_path = "./sql_lora_model.bin"

# Check if the file exists before attempting to download
if os.path.exists(model_path):
  print(f"Downloading {model_path}...")
  files.download(model_path)
  print("Download complete.")
else:
  print(f"Model file not found at {model_path}. Please ensure the training and saving steps completed successfully.")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download complete.


In [ ]:
# # prompt: code to load the model in environment to use from this "torch.save(trainer.model.state_dict(), "./sql_lora_model.bin")"

# # Load the saved LoRA weights
# lora_weights = torch.load("./sql_lora_model.bin")

# # Load the base model again (or use the one already in memory if available)
# # If 'model' variable from training is still available, you can reuse it.
# # Otherwise, load it fresh.
# # model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B").to(device) # Uncomment if needed

# # Initialize a new PEFT model with the same configuration
# # This creates a model structure with the adapter layers
# peft_model = get_peft_model(model, lora_config)

# # Load the saved state dictionary into the PEFT model
# # This applies the learned LoRA weights
# peft_model.load_state_dict(lora_weights)

# # Set the model to evaluation mode
# peft_model.eval()

# print("LoRA model loaded successfully!")

# # Now you can use 'peft_model' for inference
# # Example: create a pipeline with the loaded PEFT model
# gen_pipeline_lora = pipeline("text-generation",
#                              model=peft_model,
#                              tokenizer=tokenizer,
#                              device=device,
#                              batch_size=2, # Adjust batch size as needed
#                              max_length=50, # Adjust max_length as needed
#                              truncation=True,
#                              padding=False,
#                              return_full_text=False)

# tokenizer.padding_side = 'left' # Set padding side for inference

# # Example inference (using the first instruction from the test set)
# instruction_text_for_inference = conversation_to_text_with_template(instructions[0], tokenizer)

# with torch.no_grad():
#     generated_output_lora = gen_pipeline_lora(instruction_text_for_inference,
#                                               max_length=512, # Adjust max_length as needed for the query
#                                               num_beams=5,
#                                               early_stopping=True)

# generated_text_lora = generated_output_lora[0]["generated_text"]
# clean_sql_lora = extract_sql_from_response(generated_text_lora)

# print("\n--- Inference with Loaded LoRA Model ---")
# print('Instruction:')
# print(instructions[0]['messages'])
# print('\nExpected Response:')
# print(expected_outputs[0])
# print('\nGenerated Response (LoRA):')
# print(clean_sql_lora)
# print("---------------------------------------")

## 7. Model Evaluation with SQL-specific metrics

In [ ]:
# Redefine pipeline for fine-tuned model
gen_pipeline = pipeline("text-generation",
                        model=model,
                        tokenizer=tokenizer,
                        device=device,
                        batch_size=2,
                        max_length=50,
                        truncation=True,
                        padding=False,
                        return_full_text=False)

text_inputs = [conversation_to_text_with_template(sample, tokenizer) for sample in instructions.select(range(20))]

# Generate responses with fine-tuned model
with torch.no_grad():
    pipeline_iterator = gen_pipeline(text_inputs,
                                    max_length=512,
                                    num_beams=5,
                                    early_stopping=True)

generated_outputs_lora = []
for text in tqdm(pipeline_iterator):
  # Extract and clean the SQL from generated response
  generated_text = text[0]["generated_text"]
  clean_sql = extract_sql_from_response(generated_text)
  generated_outputs_lora.append(clean_sql)


Device set to use cuda


  0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
# SQL-specific evaluation metrics
def exact_match_score(predictions, references):
    """Calculate exact match score for SQL queries"""
    if len(predictions) != len(references):
        return 0.0

    exact_matches = 0
    for pred, ref in zip(predictions, references):
        # Normalize SQL queries (remove extra whitespace, convert to lowercase)
        pred_normalized = ' '.join(pred.lower().split())
        ref_normalized = ' '.join(ref.lower().split())

        if pred_normalized == ref_normalized:
            exact_matches += 1

    return exact_matches / len(predictions)

def sql_keyword_accuracy(predictions, references):
    """Calculate accuracy of SQL keywords"""
    sql_keywords = ['SELECT', 'FROM', 'WHERE', 'JOIN', 'GROUP BY', 'ORDER BY', 'HAVING', 'INSERT', 'UPDATE', 'DELETE']

    total_keywords = 0
    correct_keywords = 0

    for pred, ref in zip(predictions, references):
        pred_upper = pred.upper()
        ref_upper = ref.upper()

        for keyword in sql_keywords:
            if keyword in ref_upper:
                total_keywords += 1
                if keyword in pred_upper:
                    correct_keywords += 1

    return correct_keywords / total_keywords if total_keywords > 0 else 0.0


In [ ]:
# Evaluate base model
print("Base Model Evaluation:")
base_exact_match = exact_match_score(generated_outputs_base, expected_outputs[:len(generated_outputs_base)])
base_keyword_acc = sql_keyword_accuracy(generated_outputs_base, expected_outputs[:len(generated_outputs_base)])

print(f"Exact Match Score: {base_exact_match:.3f}")
print(f"SQL Keyword Accuracy: {base_keyword_acc:.3f}")

# BLEU score for reference
sacrebleu = evaluate.load("sacrebleu")
results_base = sacrebleu.compute(predictions=generated_outputs_base,
                                 references=expected_outputs[:len(generated_outputs_base)])
print(f"BLEU Score: {results_base['score']:.1f}")

print("\n" + "="*50 + "\n")


Base Model Evaluation:
Exact Match Score: 0.000
SQL Keyword Accuracy: 0.983
BLEU Score: 22.8




In [ ]:
# Evaluate fine-tuned model
print("Fine-tuned Model Evaluation:")
lora_exact_match = exact_match_score(generated_outputs_lora, expected_outputs[:len(generated_outputs_lora)])
lora_keyword_acc = sql_keyword_accuracy(generated_outputs_lora, expected_outputs[:len(generated_outputs_lora)])

print(f"Exact Match Score: {lora_exact_match:.3f}")
print(f"SQL Keyword Accuracy: {lora_keyword_acc:.3f}")

# BLEU score for fine-tuned model
results_lora = sacrebleu.compute(predictions=generated_outputs_lora,
                                 references=expected_outputs[:len(generated_outputs_lora)])
print(f"BLEU Score: {results_lora['score']:.1f}")

print("\n" + "="*50 + "\n")


Fine-tuned Model Evaluation:
Exact Match Score: 0.800
SQL Keyword Accuracy: 1.000
BLEU Score: 98.0




In [ ]:
# Show improvement
print("Improvement:")
print(f"Exact Match: {(lora_exact_match - base_exact_match):.3f}")
print(f"SQL Keyword Accuracy: {(lora_keyword_acc - base_keyword_acc):.3f}")
print(f"BLEU Score: {(results_lora['score'] - results_base['score']):.1f}")


Improvement:
Exact Match: 0.800
SQL Keyword Accuracy: 0.017
BLEU Score: 75.2


In [ ]:
# Show sample comparisons
print("\nSample Comparisons:")
for i in range(min(5, len(generated_outputs_base))):
    print(f"\n--- Example {i+1} ---")
    print(f"Expected: {expected_outputs[i]}")
    print(f"Base Model: {generated_outputs_base[i]}")
    print(f"Fine-tuned: {generated_outputs_lora[i]}")


Sample Comparisons:

--- Example 1 ---
Expected: SELECT record FROM table_name_2 WHERE attendance = "19,887"
Base Model: SELECT score FROM table_name_2 WHERE attendance = '19,887';
Fine-tuned: SELECT record FROM table_name_2 WHERE attendance = "19,887"

--- Example 2 ---
Expected: SELECT pba_team FROM table_name_41 WHERE pick < 11 AND player = "roberto jabar"
Base Model: Okay, let's see. The user is asking for the PBA team of Roberto Jabar who was picked before number 11. First, I need to figure out how to translate this into a SQL query.

Looking at the SCHEMA provided, there's a table called table_name_41 with columns pba_team, pick, and player. The columns are named pba_team, pick, and player. The user's question is about finding the PBA team of Roberto Jabar who was picked before number 11. 

So, I need to join the table with the player's name and the pick number. The pick column probably contains the pick numbers, and the player's name is in the player column. The user wants the 